In [1]:
import os
os.chdir("/home/mdarif/MLOPS/Wine-Quality-Prediction")
print(os.getcwd())

/home/mdarif/MLOPS/Wine-Quality-Prediction


In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass (frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    model_file: Path
    test_file: Path
    metrics_file: Path
    status_file: Path

In [3]:
from wine_quality_prediction.constants import *
from wine_quality_prediction.utils import read_yaml, create_directories

class ConfigurationManager:

    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        create_directories([config.root_dir])
        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            model_file=Path(config.model_file),
            test_file=Path(config.test_file),
            metrics_file=Path(config.metrics_file),
            status_file=Path(config.status_file)
        )

In [4]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from wine_quality_prediction.logger import get_logger, ModelEvaluationError
from wine_quality_prediction.utils import load_bin, save_json, get_size, save_status

logger = get_logger("model_evaluation", "model_evaluation.log")

In [5]:
class ModelEvaluation:
    """Handles model evaluation."""

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def load_model(self):
        """Load trained model."""

        try:
            model = load_bin(self.config.model_file)
            logger.info(f"Model loaded from: '{self.config.model_file}'")
            return model

        except Exception as e:
            logger.error(
                f"Model load failed - "
                f"path='{self.config.model_file}': {e}",
                exc_info=True
            )
            raise ModelEvaluationError("Failed to load model") from e

    def load_test_data(self):
        """load test dataset."""

        try:
            logger.info(f"Reading test file: '{self.config.test_file}'")
            logger.info(f"Test file size: '{get_size(self.config.test_file)}'")

            test = pd.read_csv(self.config.test_file)
            logger.info(
                f"Test data loaded - "
                f"shape: {test.shape}"
            )
            return test

        except Exception as e:
            logger.error(
                f"Test data load failed - "
                f"path='{self.config.test_file}': {e}",
                exc_info=True
            )
            raise ModelEvaluationError("Failed to load test data") from e

    def evaluate(self, model, test: pd.DataFrame):
        """Evaluate trained model."""

        try:
            target_column = "quality"

            x_test = test.drop(columns=[target_column])
            y_test = test[target_column]

            y_pred = model.predict(x_test)

            r2 = r2_score(y_test, y_pred)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            mae = mean_absolute_error(y_test, y_pred)

            metrics = {
                "r2": round(float(r2), 4),
                "rmse": round(float(rmse), 4),
                "mae": round(float(mae), 4)
            }
            logger.info(f"R2 Score: '{metrics['r2']}'")
            logger.info(f"RMSE: '{metrics['rmse']}'")
            logger.info(f"MAE: '{metrics['mae']}'")

            save_json(
                self.config.metrics_file,
                metrics
            )

            logger.info(f"Metrics saved at: '{self.config.metrics_file}'")
            return metrics, y_pred, y_test

        except Exception as e:
            logger.error(
                f"Model evaluation failed: {e}",
                exc_info=True
            )
            raise ModelEvaluationError("Failed to evaluate model") from e

    def run(self):
        """Run full model evaluation pipeline."""

        try:
            model = self.load_model()
            test = self.load_test_data()
            metrics, y_pred, y_test = self.evaluate(model, test)
            save_status(
                self.config.status_file,
                "model_evaluation",
                True
            )
            logger.info("Model evaluation completed successfully")
            return metrics, y_pred, y_test

        except Exception as e:
            save_status(
                self.config.status_file,
                "model_evaluation",
                False
            )
            logger.error(
                f"Unexpected error: {e}",
                exc_info=True
            )
            raise ModelEvaluationError("Model evaluation pipeline failed") from e

In [6]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    metrics, y_pred, y_test = model_evaluation.run()
    print(f"\nFinal metrics: '{metrics}'")
except Exception:
    raise

2026-06-02 08:17:40,854 - common - INFO - common.py:31 - YAML file loaded successfully: 'config/config.yaml'
2026-06-02 08:17:40,857 - common - INFO - common.py:31 - YAML file loaded successfully: 'params.yaml'
2026-06-02 08:17:40,859 - common - INFO - common.py:31 - YAML file loaded successfully: 'schema.yaml'
2026-06-02 08:17:40,862 - common - INFO - common.py:53 - Created directory at: 'artifacts'
2026-06-02 08:17:40,863 - common - INFO - common.py:53 - Created directory at: 'artifacts/model_evaluation'
2026-06-02 08:17:40,901 - common - INFO - common.py:128 - Binary file loaded successfully: 'artifacts/model_trainer/model.joblib'
2026-06-02 08:17:40,902 - model_evaluation - INFO - 517848603.py:12 - Model loaded from: 'artifacts/model_trainer/model.joblib'
2026-06-02 08:17:40,903 - model_evaluation - INFO - 517848603.py:27 - Reading test file: 'artifacts/data_transformation/test.csv'
2026-06-02 08:17:40,904 - model_evaluation - INFO - 517848603.py:28 - Test file size: '~49 KB'
2026-


Final metrics: '{'r2': 0.4581, 'rmse': 0.5491, 'mae': 0.4028}'
